<div align="right">

**Joint ICTP MARVEL College: Materials simulations in the age of AI**  
Thursday, 4 June 2026

&nbsp;


</div>

<div align="center">

# From Berry phases to hybrid Wannier centers with PythTB

</div>

This notebook is the runnable companion to the tutorial Markdown. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pythtb import Mesh, WFArray

## 2. Haldane HWCC flow

## Topological phase

The students will first compute the HWCC flow for the topological phase of the Haldane model. They will discover for themselves that the system is topological by investigating the HWCC flow, Chern number, and band inversion. 

In [ ]:
from pythtb.models import haldane

delta = 1.0  # staggered sublattice potential
t1 = -1.0  # nearest-neighbor hopping
t2 = -0.3  # complex next-nearest-neighbor hopping

hal_top = haldane(delta=delta, t1=t1, t2=t2, phi=np.pi / 2)

print(hal_top)
hal_top.visualize()

In [ ]:
k_nodes = [[0, 0], [2 / 3, 1 / 3], [1 / 2, 1 / 2], [1 / 3, 2 / 3], [0, 0]]
k_node_labels = [r"$\Gamma$", r"$K$", r"$M$", r"$K'$", r"$\Gamma$"]
orbital_idx = [0]  # project onto the first orbital (A sublattice)

hal_top.plot_bands(
    nk=500,
    k_nodes=k_nodes,
    k_node_labels=k_node_labels,
    proj_orb_idx=orbital_idx,
)

In [ ]:
mesh = Mesh(["k", "k"])
mesh.build_grid(shape=(41, 41), gamma_centered=False, k_endpoints=[True, True])

In [ ]:
wfa_top = WFArray(hal_top.lattice, mesh)
wfa_top.solve_model(hal_top)

In [ ]:
phase_top = wfa_top.berry_phase(axis_idx=0, state_idx=[0], contin=True)
xbar_top = phase_top / (2.0 * np.pi)

In [ ]:
ky = mesh.get_axis_range(1, 1)

fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
for shift in range(-1, 2):
    ax.plot(ky, xbar_top + shift, "o-", ms=3, color="tab:blue")

ax.set(xlim=(0, 1), ylim=(-0.5, 1.5), xlabel=r"$k_y$", ylabel=r"$\bar{x}(k_y)$")
ax.grid(alpha=0.25)
plt.show()

In [ ]:
C_top = wfa_top.chern_number(state_idx=[0], plane=(0, 1))

print(C_top)

## Trivial case

Next, they will tune the second-nearest-neighbor hopping to a smaller absolute value, which will drive the system into a trivial phase. They will then compute the HWCC flow and Chern number again, and see that the system is now trivial.

In [ ]:
from pythtb.models import haldane

delta = 1.0  # staggered sublattice potential
t1 = -1.0  # nearest-neighbor hopping
t2 = -0.1  # complex next-nearest-neighbor hopping

hal_triv = haldane(delta=delta, t1=t1, t2=t2, phi=np.pi / 2)

In [ ]:
k_nodes = [[0, 0], [2 / 3, 1 / 3], [1 / 2, 1 / 2], [1 / 3, 2 / 3], [0, 0]]
k_node_labels = [r"$\Gamma$", r"$K$", r"$M$", r"$K'$", r"$\Gamma$"]
orbital_idx = [0]  # project onto the first orbital (A sublattice)

hal_triv.plot_bands(
    nk=500,
    k_nodes=k_nodes,
    k_node_labels=k_node_labels,
    proj_orb_idx=orbital_idx,
)

In [ ]:
wfa_triv = WFArray(hal_triv.lattice, mesh)
wfa_triv.solve_model(hal_triv)

In [ ]:
phase_triv = wfa_triv.berry_phase(axis_idx=0, state_idx=[0], contin=True)
xbar_triv = phase_triv / (2.0 * np.pi)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
for shift in range(-1, 2):
    ax.plot(ky, xbar_triv + shift, "o-", ms=3, color="tab:blue")

ax.set(xlim=(0, 1), ylim=(-0.5, 1.5), xlabel=r"$k_y$", ylabel=r"$\bar{x}(k_y)$")
ax.grid(alpha=0.25)
plt.show()

In [ ]:
C_triv = wfa_triv.chern_number(state_idx=[0], plane=(0, 1))

print(C_triv)


| $t_2$ | HWCC endpoint change $\Delta\bar{x}$ | Occupied Chern number | Band inversion at $K$ or $K'$? | Topological or trivial? |
| --- | --- | --- | --- | --- |
| $-0.3$ <br><br> | -1 | 1 | Yes at $K$ | Topological |
|  $-0.1$ <br><br> | 0 | 0 | No | Trivial |

## 3. Kane-Mele Wilson-loop eigenphases


In [ ]:
from pythtb.models import kane_mele

delta = 1.0  # staggered sublattice potential
t = 1.0  # nearest-neighbor hopping
soc = 0.3  # intrinsic SOC
rashba = 0.25  # Rashba SOC

km_top = kane_mele(delta=delta, t=t, soc=soc, rashba=rashba)

print(km_top)
km_top.visualize()

In [ ]:
k_nodes = [[0, 0], [2 / 3, 1 / 3], [1 / 2, 1 / 2], [1 / 3, 2 / 3], [0, 0]]
k_node_labels = [r"$\Gamma$", r"$K$", r"$M$", r"$K'$", r"$\Gamma$"]
proj_orb_idx = [0]  # project onto the first orbital (A sublattice, spin up)

km_top.plot_bands(
    nk=500,
    k_nodes=k_nodes,
    k_node_labels=k_node_labels,
    proj_orb_idx=proj_orb_idx,
)

In [ ]:
mesh = Mesh(["k", "k"])
mesh.build_grid(shape=(41, 41))
wfa_top = WFArray(km_top.lattice, mesh, spinful=True)
wfa_top.solve_model(km_top)

centers_top = wfa_top.berry_phase(
    axis_idx=0,
    state_idx=[0, 1],
    contin=True,
    berry_evals=True,
) / (2.0 * np.pi)

In [ ]:
ky = mesh.get_axis_range(1, 1)

fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
for shift in range(-1, 3):
    ax.plot(ky, centers_top[:, 0] + shift, "k.", ms=4)
    ax.plot(ky, centers_top[:, 1] + shift, "k.", ms=4)

ax.axvline(0.5, color="tab:red", lw=0.8)
ax.set_title(rf"Kane-Mele: $\delta={delta}$")
ax.set(xlim=(0, 1), ylim=(-1, 2), xlabel=r"$k_y$", ylabel=r"$\bar{x}_n(k_y)$")
ax.grid(alpha=0.25)
plt.show()

## Trivial phase

In [ ]:
from pythtb.models import kane_mele

delta = 3.0  # staggered sublattice potential
t = 1.0  # nearest-neighbor hopping
soc = 0.3  # intrinsic SOC
rashba = 0.25  # Rashba SOC

km_triv = kane_mele(delta=delta, t=t, soc=soc, rashba=rashba)

In [ ]:
km_triv.plot_bands(
    nk=500,
    k_nodes=k_nodes,
    k_node_labels=k_node_labels,
    proj_orb_idx=proj_orb_idx,
)

In [ ]:
wfa_triv = WFArray(km_triv.lattice, mesh, spinful=True)
wfa_triv.solve_model(km_triv)

centers_triv = wfa_triv.berry_phase(
    axis_idx=0,
    state_idx=[0, 1],
    contin=True,
    berry_evals=True,
) / (2.0 * np.pi)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
for shift in range(-1, 3):
    ax.plot(ky, centers_triv[:, 0] + shift, "k.", ms=4)
    ax.plot(ky, centers_triv[:, 1] + shift, "k.", ms=4)

ax.axvline(0.5, color="tab:red", lw=0.8)
ax.set_title(rf"Kane-Mele: $\delta={delta}$")
ax.set(xlim=(0, 1), ylim=(-1, 2), xlabel=r"$k_y$", ylabel=r"$\bar{x}_n(k_y)$")
ax.grid(alpha=0.25)
plt.show()

| `delta` | Partner switching? | Band inversion? | $\quad\quad \mathbb{Z}_2 \quad\quad$ | Topological or trivial? |
| --- | :-: | :-: | :-: | :-: |
| $2.5$ <br><br>| Yes | Yes | odd | Topological |
| $3.0$ <br><br>  | No | No | even | Trivial |